# Fuzzy search vs. semantic search in Elasticsearch

Companion notebook for [Fuzzy search in Elasticsearch: does typo tolerance still earn its keep next to semantic search?](https://www.elastic.co/search-labs/blog/elasticsearch-fuzzy-search-vs-semantic-search). It reproduces the test bench and the measurements; the explanations are in the article.

**Requirements:** Elastic Cloud Hosted 9.4+ (9.5 for `FUSE`) with the Elastic Inference Service, the `analysis-phonetic` plugin, a license that includes the `rrf` and `text_similarity_reranker` retrievers, about 1.5 GB of free disk, and a `.env` copied from `.env.example`.

**Scale:** `NOTEBOOK_SCALE=demo` (default) builds about 45,000 documents in about 20 minutes; `article` builds the full million (about 80 minutes of loading). EIS bills the embeddings per million tokens.

API key, from Kibana **Dev Tools**:

```
POST /_security/api_key
{
  "name": "fuzzy-vs-semantic-notebook",
  "role_descriptors": {
    "typo-notebook": {
      "cluster": ["monitor", "manage_search_synonyms", "monitor_inference"],
      "indices": [{ "names": ["typo-*"], "privileges": ["all"] }]
    }
  }
}
```

In [ ]:
%pip install -q -r requirements.txt

## 1. Connect

Checks the phonetic plugin, both Jina endpoints, and that an existing `typo-catalog` was created by this notebook.

In [ ]:
import collections
import hashlib
import json
import math
import os
import random
import re
import statistics
import time
import zipfile
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import pyarrow.parquet as pq
import requests
from dotenv import load_dotenv
from elasticsearch import ApiError, Elasticsearch
from metaphone import doublemetaphone

load_dotenv()

ES_URL = os.environ["ELASTICSEARCH_URL"]
API_KEY = os.environ["ELASTICSEARCH_API_KEY"]
SCALE = os.getenv("NOTEBOOK_SCALE", "demo")
assert SCALE in ("demo", "article"), "NOTEBOOK_SCALE must be demo or article"

# Names of everything this notebook creates. Clean up deletes exactly these.
INDEX = "typo-catalog"
SYNONYMS_SET = "typo-misspellings"
NOTEBOOK_TAG = "fuzzy-vs-semantic-notebook"  # stored in the index _meta

EMBEDDING_ID = (
    ".jina-embeddings-v5-text-small"  # default semantic_text endpoint on ECH 9.4+
)
RERANK_ID = ".jina-reranker-v3.5"  # recommended rerank endpoint
NUM_CANDIDATES = 10_000  # the largest value the knn query accepts

es = Elasticsearch(
    ES_URL, api_key=API_KEY, request_timeout=600, max_retries=5, retry_on_timeout=True
)
print(f"elasticsearch {es.info()['version']['number']} | scale={SCALE}")

plugins = {
    p["name"]
    for node in es.nodes.info(metric="plugins")["nodes"].values()
    for p in node["plugins"]
}
assert (
    "analysis-phonetic" in plugins
), "Enable the analysis-phonetic extension on the deployment first"
for endpoint in (EMBEDDING_ID, RERANK_ID):
    ep = es.inference.get(inference_id=endpoint)["endpoints"][0]
    print(endpoint, ep["task_type"])


def index_is_ours():
    """None if the index doesn't exist, True if this notebook created it, False otherwise."""
    if not es.indices.exists(index=INDEX):
        return None
    meta = es.indices.get_mapping(index=INDEX)[INDEX]["mappings"].get("_meta", {})
    return meta.get("created_by") == NOTEBOOK_TAG


assert (
    index_is_ours() is not False
), f"{INDEX} already exists and wasn't created by this notebook: delete it or rename INDEX"

## 2. Download the source data

| Source | License |
| --- | --- |
| [Amazon Shopping Queries Dataset (ESCI)](https://github.com/amazon-science/esci-data) | Apache 2.0 |
| [openFDA NDC directory](https://open.fda.gov/data/downloads/) | Public domain |
| [US Census 2010 surnames](https://www.census.gov/topics/population/genealogy/data/2010_surnames.html) and [1990 first names](https://www.census.gov/topics/population/genealogy/data/1990_census.html) | Public domain |
| [Wikipedia: Lists of common misspellings/For machines](https://en.wikipedia.org/wiki/Wikipedia:Lists_of_common_misspellings/For_machines) | CC BY-SA 4.0 |

Cached in `nb_data/src`. openFDA and Wikipedia change over time, so counts can differ slightly from the article.

In [ ]:
SRC = Path("nb_data/src")
SRC.mkdir(parents=True, exist_ok=True)
HEADERS = {"User-Agent": "elastic-labs-typo-notebook/1.0"}
ESCI = "https://github.com/amazon-science/esci-data/raw/main/shopping_queries_dataset/"
FILES = {
    "shopping_queries_dataset_products.parquet": ESCI
    + "shopping_queries_dataset_products.parquet",
    "shopping_queries_dataset_examples.parquet": ESCI
    + "shopping_queries_dataset_examples.parquet",
    "drug-ndc.json.zip": "https://download.open.fda.gov/drug/ndc/drug-ndc-0001-of-0001.json.zip",
    "names.zip": "https://www2.census.gov/topics/genealogy/2010surnames/names.zip",
    "dist.male.first": "https://www2.census.gov/topics/genealogy/1990surnames/dist.male.first",
    "dist.female.first": "https://www2.census.gov/topics/genealogy/1990surnames/dist.female.first",
    "formachines.json": "https://en.wikipedia.org/w/api.php?action=parse&format=json&prop=wikitext"
    "&page=Wikipedia:Lists_of_common_misspellings/For_machines",
}


def download(name, url):
    path = SRC / name
    if path.exists():
        return path
    with requests.get(url, headers=HEADERS, stream=True, timeout=300) as r:
        r.raise_for_status()
        with open(path.with_suffix(".part"), "wb") as f:
            for block in r.iter_content(1 << 20):
                f.write(block)
    path.with_suffix(".part").rename(path)
    return path


for name, url in FILES.items():
    print(f"{name:45} {download(name, url).stat().st_size / 2**20:8,.1f} MB")

for archive, member in (
    ("drug-ndc.json.zip", "drug-ndc-0001-of-0001.json"),
    ("names.zip", "Names_2010Census.csv"),
):
    if not (SRC / member).exists():
        zipfile.ZipFile(SRC / archive).extract(member, SRC)

## 3. Build the test bench

![Test bench: sources, the typo-catalog index, judged queries and metrics](images/test-bench.jpg)

In [ ]:
SIZES = {
    "demo": dict(
        products=25_000,
        medications=10_000,
        customers=10_000,
        sku_asin=67,
        sku_ndc=33,
        drug=100,
        surname=100,
        soundalike=75,
        phrase_per_half=50,
        exploratory=100,
        neg_sku=33,
        neg_drug=33,
        neg_surname=33,
    ),
    "article": dict(
        products=850_000,
        medications=None,
        customers=100_000,
        sku_asin=200,
        sku_ndc=100,
        drug=200,
        surname=200,
        soundalike=150,
        phrase_per_half=100,
        exploratory=300,
        neg_sku=100,
        neg_drug=60,
        neg_surname=100,
    ),
}[SCALE]
rng = random.Random(42)


def half(word):
    return (
        "train" if int(hashlib.sha1(word.encode()).hexdigest(), 16) % 2 == 0 else "test"
    )


wikitext = json.load(open(SRC / "formachines.json"))["parse"]["wikitext"]["*"]
misspellings = {}
for line in wikitext.splitlines():
    if "->" in line:
        wrong, right = (x.strip() for x in line.split("->", 1))
        if re.fullmatch(r"[a-z]+", wrong) and re.fullmatch(r"[a-z]+", right):
            misspellings[wrong] = right
misspellings.pop("thru", None)  # an accepted informal spelling, not a typo
n_train = sum(half(r) == "train" for r in misspellings.values())
print(
    f"misspelling pairs {len(misspellings):,}: {n_train:,} go to the synonyms set, {len(misspellings) - n_train:,} stay out"
)

# ESCI judged queries: real shopper queries with a listed misspelling, and exploratory queries.
ex = pq.read_table(
    SRC / "shopping_queries_dataset_examples.parquet",
    columns=["query_id", "query", "product_id", "esci_label", "product_locale"],
    filters=[("product_locale", "==", "us")],
).to_pandas()
ex["grade"] = ex.esci_label.map({"E": 2, "S": 1, "C": 0, "I": 0})
real_typo, exploratory_pool = [], []
for (qid, q), g in ex.groupby(["query_id", "query"]):
    toks = re.findall(r"[a-z]+", q.lower())
    rated = {pid: int(gr) for pid, gr in zip(g.product_id, g.grade) if gr > 0}
    if not rated:
        continue
    item = {"query_id": int(qid), "query": q, "ratings": rated}
    if any(t in misspellings for t in toks):
        real_typo.append(item)
    elif sum(v == 2 for v in rated.values()) >= 3 and len(toks) >= 3:
        exploratory_pool.append(item)
exploratory = rng.sample(exploratory_pool, SIZES["exploratory"])
judged = {pid for it in real_typo + exploratory for pid in it["ratings"]}
print(
    f"US queries in ESCI {ex.query_id.nunique():,} | real typo queries {len(real_typo)} | "
    f"exploratory {len(exploratory)} | judged products {len(judged):,}"
)

In [ ]:
# Products: every judged product, plus a random sample up to the target size.
prod = pq.read_table(
    SRC / "shopping_queries_dataset_products.parquet",
    columns=["product_id", "product_title", "product_brand", "product_locale"],
    filters=[("product_locale", "==", "us")],
).to_pandas()
prod = prod.dropna(subset=["product_title"]).drop_duplicates("product_id")
keep = prod.product_id.isin(judged)
fill = min(max(SIZES["products"] - int(keep.sum()), 0), int((~keep).sum()))
prod = pd.concat([prod[keep], prod[~keep].sample(fill, random_state=7)])

# Medications: one record per brand, generic, strength and form.
meds = {}
for r in json.load(open(SRC / "drug-ndc-0001-of-0001.json"))["results"]:
    brand, generic = (r.get("brand_name") or "").strip(), (
        r.get("generic_name") or ""
    ).strip()
    if not brand or not generic or not r.get("product_ndc"):
        continue
    strength = ", ".join(
        a.get("strength", "") for a in (r.get("active_ingredients") or [])[:2]
    )
    form = (r.get("dosage_form") or "").title()
    meds.setdefault(
        (brand.lower(), generic.lower(), strength.lower(), form.lower()),
        {
            "sku": r["product_ndc"],
            "name": f"{brand.title()} ({generic.lower()}) {strength} {form}".strip(),
            "drug_brand": brand.lower(),
            "drug_generic": generic.lower(),
        },
    )
meds = list(meds.values())
singles = collections.Counter(
    m["drug_brand"] for m in meds if re.fullmatch(r"[a-z]{7,}", m["drug_brand"])
)
holdout_drugs = set(rng.sample(sorted(b for b, n in singles.items() if n == 1), 60))
meds = [m for m in meds if m["drug_brand"] not in holdout_drugs]
if SIZES["medications"]:
    meds = rng.sample(meds, min(SIZES["medications"], len(meds)))

# Customers: synthetic, names sampled by frequency. A tail of real surnames is held out for negatives.
sur = pd.read_csv(SRC / "Names_2010Census.csv", keep_default_na=False)
sur = sur[sur.name != "ALL OTHER NAMES"].head(50_000)
sur["count"] = pd.to_numeric(sur["count"], errors="coerce").fillna(100)
holdout_surnames = set(rng.sample(sur.iloc[20_000:].name.tolist(), 300))
sur = sur[~sur.name.isin(holdout_surnames)]
firsts = [
    (p[0].title(), float(p[1]))
    for f in ("dist.male.first", "dist.female.first")
    for p in (line.split() for line in open(SRC / f))
]
f_names, f_w = zip(*firsts)
s_names, s_w = sur.name.str.title().tolist(), sur["count"].tolist()
customers = []
for i in range(SIZES["customers"]):
    first, last = rng.choices(f_names, f_w)[0], rng.choices(s_names, s_w)[0]
    customers.append(
        {
            "sku": f"C{1_000_000 + i}",
            "name": f"{first} {last}",
            "first_name": first,
            "last_name": last,
        }
    )

docs = (
    [
        {
            "_id": r.product_id,
            "doc_type": "product",
            "sku": r.product_id,
            "name": r.product_title,
        }
        for r in prod.itertuples(index=False)
    ]
    + [{"_id": f"med-{i}", "doc_type": "medication", **m} for i, m in enumerate(meds)]
    + [{"_id": c["sku"], "doc_type": "customer", **c} for c in customers]
)
for d in docs:
    d["label"] = (
        f"{d['name']} ({d['sku']})"  # what the reranker reads in the name + SKU variant
    )
print(
    f"{len(prod):,} products, {len(meds):,} medications, {len(customers):,} customers: {len(docs):,} documents"
)

# How clean are the product titles?
title_tokens = [
    set(re.findall(r"[a-z]+", d["name"].lower()))
    for d in docs
    if d["doc_type"] == "product"
]
dirty = sum(bool(t & misspellings.keys()) for t in title_tokens)
print(
    f"product titles with a listed misspelling: {dirty:,} | titles with 'rechargable': "
    f"{sum('rechargable' in t for t in title_tokens):,}"
)

Judged queries: one-edit typos, sound-alike surnames, correctly spelled twins, exploratory queries and negatives.

In [ ]:
KEYBOARD = ["qwertyuiop", "asdfghjkl", "zxcvbnm"]
NEIGHBORS = collections.defaultdict(str)
for r, row in enumerate(KEYBOARD):
    for i, ch in enumerate(row):
        for rr in (r - 1, r, r + 1):
            for ii in (i - 1, i, i + 1):
                if (
                    0 <= rr < 3
                    and 0 <= ii < len(KEYBOARD[rr])
                    and KEYBOARD[rr][ii] != ch
                ):
                    NEIGHBORS[ch] += KEYBOARD[rr][ii]


def keyboard_edit(word):
    w, op = list(word), rng.choice(["sub", "del", "ins", "swap"])
    i = rng.randrange(1, len(w) - (1 if op == "swap" else 0))
    if op == "sub":
        w[i] = rng.choice(NEIGHBORS[w[i]] or "aeiou")
    elif op == "del":
        del w[i]
    elif op == "ins":
        w.insert(i, rng.choice(NEIGHBORS[w[i]] or "aeiou"))
    else:
        w[i], w[i + 1] = w[i + 1], w[i]
    out = "".join(w)
    return out if out != word else keyboard_edit(word)


def code_edit(code):
    w, op = list(code), rng.choice(["sub", "del", "ins", "swap"])
    pos = [i for i, ch in enumerate(w) if ch.isalnum()]
    i = rng.choice(pos[:-1] if op == "swap" else pos)
    pool = "0123456789" if w[i].isdigit() else "ABCDEFGHIJKLMNOPQRSTUVWXYZ"
    if op == "sub":
        w[i] = rng.choice(pool.replace(w[i], ""))
    elif op == "del":
        del w[i]
    elif op == "ins":
        w.insert(i, rng.choice(pool))
    else:
        w[i], w[i + 1] = w[i + 1], w[i]
    out = "".join(w)
    return out if out != code else code_edit(code)


def lev(a, b):
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (ca != cb)))
        prev = cur
    return prev[-1]


TOK = re.compile(r"[a-z0-9]+")
sets = collections.defaultdict(list)


def add(set_name, query, relevant, **meta):
    sets[set_name].append(
        {
            "id": f"{set_name}-{len(sets[set_name])}",
            "query": query,
            "ratings": {r: 1 for r in relevant},
            **meta,
        }
    )


products = [d for d in docs if d["doc_type"] == "product"]
medications = [d for d in docs if d["doc_type"] == "medication"]
custs = [d for d in docs if d["doc_type"] == "customer"]
all_skus = {d["sku"].lower() for d in docs}

# SKUs: ASINs and NDC codes with one edit.
for pool, n, kind in (
    (products, SIZES["sku_asin"], "asin"),
    (medications, SIZES["sku_ndc"], "ndc"),
):
    picked = 0
    for d in rng.sample(pool, min(len(pool), n * 3)):
        if picked == n:
            break
        if kind == "asin" and not re.fullmatch(r"B0[0-9A-Z]{8}", d["sku"]):
            continue
        typo = code_edit(d["sku"])
        if typo.lower() not in all_skus:
            add("sku_typo", typo, [d["_id"]], original=d["sku"], kind=kind)
            add("sku_exact", d["sku"], [d["_id"]], kind=kind)
            picked += 1

# Drug names with one keyboard edit.
med_by_token = collections.defaultdict(set)
for d in medications:
    for t in set(TOK.findall(d["name"].lower())):
        med_by_token[t].add(d["_id"])
drug_words = sorted(
    {
        w
        for d in medications
        for w in (d["drug_brand"], d["drug_generic"])
        if re.fullmatch(r"[a-z]{6,}", w) and 1 <= len(med_by_token[w]) <= 50
    }
)
for w in rng.sample(drug_words, min(SIZES["drug"], len(drug_words))):
    typo = keyboard_edit(w)
    if typo not in med_by_token:
        add("drug_typo", typo, med_by_token[w], original=w)
        add("drug_exact", w, med_by_token[w])

# Surnames with one keyboard edit, and sound-alike surnames.
by_full = collections.defaultdict(set)
for d in custs:
    by_full[(d["first_name"], d["last_name"])].add(d["_id"])
surnames = {d["last_name"] for d in custs}
for first, last in rng.sample([k for k in by_full if len(k[1]) >= 5], SIZES["surname"]):
    typo = keyboard_edit(last.lower()).title()
    if typo not in surnames:
        add("surname_typo", f"{first} {typo}", by_full[(first, last)], original=last)
        add("surname_exact", f"{first} {last}", by_full[(first, last)])

census_surnames = sorted(set(s_names) | surnames)
by_code = collections.defaultdict(list)
for s in census_surnames:
    by_code[doublemetaphone(s)].append(s)
pairs = [
    (a, b)
    for group in by_code.values()
    for a in group
    for b in group
    if a != b
    and b in surnames
    and a[0] == b[0]
    and 2 <= lev(a.lower(), b.lower()) <= 3
    and abs(len(a) - len(b)) <= 1
    and len(a) >= 5
]
rng.shuffle(pairs)
used = set()
for written, actual in pairs:
    if len(sets["surname_soundalike"]) == SIZES["soundalike"]:
        break
    targets = [k for k in by_full if k[1] == actual and (k[0], written) not in by_full]
    if targets and actual not in used:
        used.add(actual)
        first = rng.choice(targets)[0]
        add(
            "surname_soundalike",
            f"{first} {written}",
            by_full[(first, actual)],
            original=actual,
            written=written,
        )

# Wikipedia misspellings inside two-word phrases taken from product titles.
right_to_wrong = collections.defaultdict(list)
for w, r in misspellings.items():
    right_to_wrong[r].append(w)
posting = collections.defaultdict(set)
for d in products:
    for t in set(TOK.findall(d["name"].lower())):
        posting[t].add(d["_id"])
title = {d["_id"]: d["name"] for d in products}
for split in ("train", "test"):
    words = [w for w in right_to_wrong if half(w) == split and len(posting[w]) >= 3]
    rng.shuffle(words)
    made = 0
    for w in words:
        if made == SIZES["phrase_per_half"]:
            break
        wrong = rng.choice(right_to_wrong[w])
        if lev(wrong, w) > 3 or wrong in posting:
            continue
        toks = TOK.findall(title[rng.choice(sorted(posting[w]))].lower())
        i = toks.index(w)
        other = [
            t
            for t in (
                toks[i - 1] if i > 0 else None,
                toks[i + 1] if i + 1 < len(toks) else None,
            )
            if t and t.isalpha() and len(t) > 3 and t != w
        ]
        if not other or not 1 <= len(posting[w] & posting[other[0]]) <= 100:
            continue
        o = other[0]
        q = [o, wrong] if toks.index(o) < i else [wrong, o]
        add("word_typo", " ".join(q), posting[w] & posting[o], split=split)
        add(
            "word_exact",
            " ".join(x if x != wrong else w for x in q),
            posting[w] & posting[o],
            split=split,
        )
        made += 1

# Real ESCI queries: human labels, exact = 2, substitute = 1.
for it in real_typo:
    sets["word_typo_real"].append(
        {
            "id": f"word_typo_real-{it['query_id']}",
            "query": it["query"],
            "ratings": it["ratings"],
        }
    )
for it in exploratory:
    sets["exploratory"].append(
        {
            "id": f"exploratory-{it['query_id']}",
            "query": it["query"],
            "ratings": it["ratings"],
        }
    )

# Negatives: SKUs that don't exist, held-out drug names and held-out surnames.
while sum(q["kind"] == "sku" for q in sets["negative"]) < SIZES["neg_sku"]:
    s = "B0" + "".join(
        rng.choice("0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ") for _ in range(8)
    )
    if s.lower() not in all_skus:
        add("negative", s, [], kind="sku")
for w in sorted(holdout_drugs)[: SIZES["neg_drug"]]:
    add("negative", w, [], kind="drug")
first_names = sorted({d["first_name"] for d in custs})
for s in rng.sample(sorted(holdout_surnames), SIZES["neg_surname"]):
    add("negative", f"{rng.choice(first_names)} {s.title()}", [], kind="surname")

counts = {k: len(v) for k, v in sets.items()}
print(counts)
print("total judged queries:", sum(counts.values()))

## 4. Create the synonyms set and the index

Every mechanism is a multi-field of `name` or `sku`.

In [ ]:
rules = [
    {"id": f"r{i}", "synonyms": f"{w} => {r}"}
    for i, (w, r) in enumerate(
        sorted((w, r) for w, r in misspellings.items() if half(r) == "train")
    )
]
es.synonyms.put_synonym(id=SYNONYMS_SET, synonyms_set=rules)
print("synonym rules", len(rules))

settings = {
    "number_of_shards": 1,
    "analysis": {
        "tokenizer": {
            "autocomplete": {
                "type": "edge_ngram",
                "min_gram": 2,
                "max_gram": 10,
                "token_chars": ["letter", "digit"],
            }
        },
        "filter": {
            "dm_phonetic": {
                "type": "phonetic",
                "encoder": "double_metaphone",
                "replace": True,
            },
            "misspellings": {
                "type": "synonym_graph",
                "synonyms_set": SYNONYMS_SET,
                "updateable": True,
            },
        },
        "analyzer": {
            "autocomplete": {"tokenizer": "autocomplete", "filter": ["lowercase"]},
            "autocomplete_search": {"tokenizer": "standard", "filter": ["lowercase"]},
            "phonetic": {
                "tokenizer": "standard",
                "filter": ["lowercase", "dm_phonetic"],
            },
            "sku_exact": {"tokenizer": "keyword", "filter": ["lowercase"]},
            "with_misspellings": {
                "tokenizer": "standard",
                "filter": ["lowercase", "misspellings"],
            },
        },
    },
}
mappings = {
    "_meta": {"created_by": NOTEBOOK_TAG},
    "properties": {
        "doc_type": {"type": "keyword"},
        "sku": {
            "type": "keyword",
            "copy_to": ["semantic_dense"],
            "fields": {
                "text": {"type": "text", "analyzer": "sku_exact"},
                "edge": {
                    "type": "text",
                    "analyzer": "autocomplete",
                    "search_analyzer": "autocomplete_search",
                },
            },
        },
        "name": {
            "type": "text",
            "copy_to": ["semantic_dense"],
            "fields": {
                "edge": {
                    "type": "text",
                    "analyzer": "autocomplete",
                    "search_analyzer": "autocomplete_search",
                },
                "phonetic": {"type": "text", "analyzer": "phonetic"},
            },
        },
        "suggest": {"type": "completion", "analyzer": "standard"},
        "label": {"type": "text", "index": False},
        "semantic_dense": {"type": "semantic_text", "inference_id": EMBEDDING_ID},
    },
}

if index_is_ours():
    es.indices.delete(index=INDEX)
es.indices.create(index=INDEX, settings=settings, mappings=mappings)
print("created", INDEX)

## 5. Load the documents with `_bulk`

EIS embeds each value inside the bulk request, so it sets the pace.

In [ ]:
def suggest_inputs(d):
    if d["doc_type"] == "customer":
        return [d["name"], f"{d['last_name']} {d['first_name']}", d["sku"]]
    if d["doc_type"] == "medication":
        return [d["name"], d["drug_generic"], d["sku"]]
    return [d["name"][:50], d["sku"]]


def bulk_chunk(chunk):
    pending = chunk
    for attempt in range(10):
        ops = []
        for d in pending:
            src = {k: v for k, v in d.items() if k != "_id"}
            src["suggest"] = {"input": suggest_inputs(d)}
            ops += [{"index": {"_index": INDEX, "_id": d["_id"]}}, src]
        try:
            resp = es.bulk(operations=ops)
        except ApiError as e:
            if e.meta.status not in (429, 502, 503, 504):
                raise
            time.sleep(min(2**attempt, 120))
            continue
        retryable = (429, 500, 502, 503, 504)
        retry = [
            d
            for d, item in zip(pending, resp["items"])
            if item["index"]["status"] in retryable
        ]
        failed = [
            item["index"]
            for item in resp["items"]
            if item["index"]["status"] >= 300
            and item["index"]["status"] not in retryable
        ]
        if failed:
            raise RuntimeError(failed[:3])
        if not retry:
            return len(chunk)
        pending = retry
        time.sleep(min(2**attempt, 120))
    raise RuntimeError(f"{len(pending)} documents still rejected after retries")


t0 = time.time()
chunks = [docs[i : i + 250] for i in range(0, len(docs), 250)]
done, step = 0, max(len(docs) // 20, 250)
with ThreadPoolExecutor(6) as pool:
    for n in pool.map(bulk_chunk, chunks):
        done += n
        if done % step < 250:
            print(f"{done:,} documents, {done / (time.time() - t0):,.0f}/s")
es.indices.refresh(index=INDEX)
elapsed = time.time() - t0
print(
    f"{es.count(index=INDEX)['count']:,} documents in {INDEX} after {elapsed / 60:,.1f} min "
    f"({len(docs) / elapsed:,.0f} documents/s)"
)

What each field stores, with `_analyze`:

![Double metaphone indexes a sound code instead of the spelling](images/double-metaphone.jpg)

In [ ]:
def analyze(analyzer, text):
    return [
        t["token"]
        for t in es.indices.analyze(index=INDEX, analyzer=analyzer, text=text)["tokens"]
    ]


for analyzer, text in [
    ("standard", "Bonnie Fountain"),
    ("autocomplete", "Bonnie Fountain"),
    ("phonetic", "Bonnie Fountain"),
    ("phonetic", "Bonnie Fontana"),
    ("phonetic", "Fontana fountain pen"),
    ("phonetic", "B07TXLWJB2 B07TJLWJB2"),
    ("with_misspellings", "bose wireless reciever"),
]:
    print(f"{analyzer:18} {text!r:24} -> {analyze(analyzer, text)}")

## 6. Score with `_rank_eval`

`_rank_eval` doesn't accept retrievers, so RRF and the completion suggester are scored in Python with the same formulas, and the MRR of both is compared.

In [ ]:
FIELDS = ["name", "sku.text"]


def strict(q):
    return {"query": {"multi_match": {"query": q, "fields": FIELDS, "operator": "and"}}}


def fuzzy(q):
    return {
        "query": {
            "multi_match": {
                "query": q,
                "fields": FIELDS,
                "operator": "and",
                "fuzziness": "AUTO",
            }
        }
    }


def edge_ngram(q):
    return {
        "query": {
            "multi_match": {
                "query": q,
                "fields": ["name.edge", "sku.edge"],
                "operator": "and",
            }
        }
    }


def phonetic(q):
    return {"query": {"match": {"name.phonetic": {"query": q, "operator": "and"}}}}


def synonyms(q):
    return {
        "query": {
            "bool": {
                "should": [
                    {
                        "match": {
                            "name": {
                                "query": q,
                                "operator": "and",
                                "analyzer": "with_misspellings",
                            }
                        }
                    },
                    {"match": {"sku.text": q}},
                ]
            }
        }
    }


def semantic_match(q):
    return {"query": {"match": {"semantic_dense": q}}}


def semantic_knn(q):
    return {
        "query": {
            "knn": {
                "field": "semantic_dense",
                "k": 10,
                "num_candidates": NUM_CANDIDATES,
                "query_vector_builder": {"text_embedding": {"model_text": q}},
            }
        }
    }


def blend(q):
    # Exact terms score in both clauses; fuzzy-only matches fill the gaps at a lower boost.
    return {
        "query": {
            "bool": {
                "should": [
                    {
                        "multi_match": {
                            "query": q,
                            "fields": FIELDS,
                            "operator": "and",
                            "boost": 2,
                        }
                    },
                    {
                        "multi_match": {
                            "query": q,
                            "fields": FIELDS,
                            "operator": "and",
                            "fuzziness": "AUTO",
                            "boost": 0.5,
                        }
                    },
                ]
            }
        }
    }


def rrf(q):
    return {
        "retriever": {
            "rrf": {
                "retrievers": [
                    {"standard": {"query": blend(q)["query"]}},
                    {"standard": {"query": semantic_knn(q)["query"]}},
                ],
                "rank_window_size": 50,
            }
        }
    }


def reranked(field):
    def config(q):
        return {
            "retriever": {
                "text_similarity_reranker": {
                    "retriever": rrf(q)["retriever"],
                    "field": field,
                    "inference_id": RERANK_ID,
                    "inference_text": q,
                    "rank_window_size": 50,
                }
            }
        }

    return config


CONFIGS = {
    "strict": strict,
    "fuzzy": fuzzy,
    "edge_ngram": edge_ngram,
    "phonetic": phonetic,
    "synonyms": synonyms,
    "semantic_match": semantic_match,
    "semantic_knn": semantic_knn,
    "blend": blend,
    "rrf": rrf,
    "rrf_rerank_name": reranked("name"),
    "rrf_rerank": reranked("label"),
}
RETRIEVER_CONFIGS = {"rrf", "rrf_rerank_name", "rrf_rerank"}
K = 10
queries = [(s, q) for s, qs in sets.items() for q in qs]
rank_eval_scores = {}


def retry(fn, attempts=6):
    for attempt in range(attempts):
        try:
            return fn()
        except ApiError as e:
            if (
                e.meta.status not in (429, 500, 502, 503, 504)
                or attempt == attempts - 1
            ):
                raise
            time.sleep(min(2**attempt, 60))


def rank_eval_hits(cfg, batch=50):
    # Returns {query id: [top-10 ids]} for a Query DSL configuration, and keeps _rank_eval's score per query.
    out, scores = {}, {}
    for i in range(0, len(queries), batch):
        part = [q for _, q in queries[i : i + batch]]
        for attempt in range(8):
            reqs = [
                {
                    "id": q["id"],
                    "request": CONFIGS[cfg](q["query"]),
                    "ratings": [
                        {"_index": INDEX, "_id": d, "rating": r}
                        for d, r in q["ratings"].items()
                    ],
                }
                for q in part
                if q["id"] not in out
            ]
            resp = retry(
                lambda: es.rank_eval(
                    index=INDEX,
                    requests=reqs,
                    metric={
                        "mean_reciprocal_rank": {"k": K, "relevant_rating_threshold": 1}
                    },
                )
            )
            for qid, d in resp["details"].items():
                out[qid] = [h["hit"]["_id"] for h in d["hits"]]
                scores[qid] = d["metric_score"]
            if not resp.get("failures"):
                break
            time.sleep(
                min(2**attempt, 60)
            )  # failed requests (often EIS rate limits) are retried
        else:
            raise RuntimeError(list(resp["failures"].items())[:2])
    rank_eval_scores[cfg] = scores
    return out


def search_one(cfg, q, size=K):
    if cfg == "completion":
        r = retry(
            lambda: es.search(
                index=INDEX,
                source=False,
                suggest={
                    "s": {
                        "prefix": q,
                        "completion": {
                            "field": "suggest",
                            "size": size,
                            "fuzzy": {"fuzziness": "AUTO"},
                        },
                    }
                },
            )
        )
        return r, [o["_id"] for o in r["suggest"]["s"][0]["options"]]
    r = retry(
        lambda: es.search(index=INDEX, size=size, source=False, **CONFIGS[cfg](q))
    )
    return r, [h["_id"] for h in r["hits"]["hits"]]


def search_hits(cfg):
    with ThreadPoolExecutor(4) as pool:
        return dict(
            pool.map(
                lambda item: (item[1]["id"], search_one(cfg, item[1]["query"])[1]),
                queries,
            )
        )


hits = {}
for cfg in CONFIGS:
    t0 = time.time()
    hits[cfg] = search_hits(cfg) if cfg in RETRIEVER_CONFIGS else rank_eval_hits(cfg)
    print(f"{cfg:16} {time.time() - t0:7.0f} s")
hits["completion"] = search_hits("completion")
# Fallback: strict first, fuzzy only when strict returns nothing.
hits["fallback"] = {qid: (h or hits["fuzzy"][qid]) for qid, h in hits["strict"].items()}

In [ ]:
ratings = {q["id"]: q["ratings"] for _, q in queries}
set_of = {q["id"]: s for s, q in queries}
meta = {q["id"]: q for _, q in queries}
# A negative only counts if nothing in the catalog matches it exactly (a held-out brand can still be a generic name).
leaked = {qid for qid, h in hits["strict"].items() if set_of[qid] == "negative" and h}
print(
    f"negatives dropped because the strict query finds them: {len(leaked)} "
    f"({collections.Counter(meta[q]['kind'] for q in leaked)})"
)
print(f"judged queries after dropping them: {len(queries) - len(leaked):,}")


def ndcg(h, rel):
    dcg = sum((2 ** rel.get(d, 0) - 1) / math.log2(i + 2) for i, d in enumerate(h[:K]))
    ideal = sum(
        (2**g - 1) / math.log2(i + 2)
        for i, g in enumerate(sorted(rel.values(), reverse=True)[:K])
    )
    return dcg / ideal if ideal else 0.0


def rr(h, rel):
    return next((1 / (i + 1) for i, d in enumerate(h[:K]) if rel.get(d, 0) > 0), 0.0)


def caught(cfg, qid):
    return any(ratings[qid].get(d, 0) > 0 for d in hits[cfg][qid][:K])


def wrong_first(cfg, qid):
    h = hits[cfg][qid]
    return bool(h) and ratings[qid].get(h[0], 0) == 0


rows = []
for cfg, by_q in hits.items():
    for s in sets:
        qs = [qid for qid in by_q if set_of[qid] == s and qid not in leaked]
        if s == "negative":
            rows.append(
                {
                    "config": cfg,
                    "set": s,
                    "n": len(qs),
                    "false_positives": statistics.mean(bool(by_q[q]) for q in qs),
                }
            )
            continue
        rows.append(
            {
                "config": cfg,
                "set": s,
                "n": len(qs),
                "catch": statistics.mean(caught(cfg, q) for q in qs),
                "wrong_first": statistics.mean(wrong_first(cfg, q) for q in qs),
                "ndcg": statistics.mean(ndcg(by_q[q], ratings[q]) for q in qs),
            }
        )
results = pd.DataFrame(rows)

TYPO_SETS = [
    "word_typo_real",
    "word_typo",
    "drug_typo",
    "surname_typo",
    "surname_soundalike",
    "sku_typo",
]
ORDER = [
    "strict",
    "fuzzy",
    "edge_ngram",
    "phonetic",
    "completion",
    "synonyms",
    "semantic_match",
    "semantic_knn",
    "fallback",
    "blend",
    "rrf",
    "rrf_rerank_name",
    "rrf_rerank",
]
catch = results.pivot(index="config", columns="set", values="catch").loc[
    ORDER, TYPO_SETS
]
print("Catch rate (relevant result in the top 10), %")
print((catch * 100).round(1).to_string())

# Cross-check: our mean reciprocal rank vs the one _rank_eval reports, over the six typo sets.
typo_q = [q for q in ratings if set_of[q] in TYPO_SETS]
print("\nMRR@10 over the typo sets: notebook vs _rank_eval")
for cfg, scores in rank_eval_scores.items():
    ours = statistics.mean(rr(hits[cfg][q], ratings[q]) for q in typo_q)
    theirs = statistics.mean(scores[q] for q in typo_q)
    print(f"{cfg:16} {ours:.4f} {theirs:.4f} diff {abs(ours - theirs):.4f}")

## 7. Fuzzy search and three other lexical mechanisms

![How fuzziness AUTO counts edits](images/fuzziness-auto.jpg)

![Edge n-grams and the completion suggester](images/prefix-mechanisms.jpg)

In [ ]:
neg = [q for q in ratings if set_of[q] == "negative" and q not in leaked]
print("False positives by kind of negative, %")
for cfg in ("fuzzy", "phonetic", "completion", "synonyms", "semantic_knn"):
    by_kind = {
        k: statistics.mean(bool(hits[cfg][q]) for q in neg if meta[q]["kind"] == k)
        for k in ("drug", "surname", "sku")
    }
    total = statistics.mean(bool(hits[cfg][q]) for q in neg)
    print(
        f"{cfg:14} total {total:6.1%} | "
        + " | ".join(f"{k} {v:6.1%}" for k, v in by_kind.items())
    )

sku = [q for q in ratings if set_of[q] == "sku_typo"]
for kind in ("asin", "ndc"):
    qs = [q for q in sku if meta[q]["kind"] == kind]
    first = sum(
        bool(hits["fuzzy"][q]) and ratings[q].get(hits["fuzzy"][q][0], 0) > 0
        for q in qs
    )
    print(
        f"fuzzy on {kind.upper()} typos: right document first for {first} of {len(qs)}, "
        f"in the top 10 for {sum(caught('fuzzy', q) for q in qs)} of {len(qs)}"
    )

snd = [q for q in ratings if set_of[q] == "surname_soundalike"]
short = [q for q in snd if len(meta[q]["written"]) <= 5]
long_ = [q for q in snd if len(meta[q]["written"]) > 5]
print(
    f"fuzzy on sound-alike surnames: {sum(caught('fuzzy', q) for q in short)} of {len(short)} short (5 letters or fewer), "
    f"{sum(caught('fuzzy', q) for q in long_)} of {len(long_)} longer"
)

In [ ]:
def show(resp, n=3, fields=("name", "sku")):
    total = resp["hits"]["total"]["value"] if "hits" in resp else None
    rows = [
        (
            round(h["_score"], 4) if h.get("_score") is not None else None,
            *(str(h["_source"].get(f, ""))[:60] for f in fields),
        )
        for h in resp["hits"]["hits"][:n]
    ]
    print(f"  total={total}")
    for r in rows:
        print("   ", r)


def run(title, body, n=3, fields=("name", "sku")):
    print(title)
    show(
        retry(lambda: es.search(index=INDEX, size=n, source=list(fields), **body)),
        n,
        fields,
    )


run(
    "strict: 312 rechargable hearing aid batteries",
    strict("312 rechargable hearing aid batteries"),
)
run(
    "fuzzy: 312 rechargable hearing aid batteries",
    fuzzy("312 rechargable hearing aid batteries"),
)
run(
    "fuzzy on sku.text: B07TJLWJB2",
    {"query": {"match": {"sku.text": {"query": "B07TJLWJB2", "fuzziness": "AUTO"}}}},
)
run("fuzzy: celexa", fuzzy("celexa"), fields=("name", "sku", "doc_type"))

r = retry(
    lambda: es.search(
        index=INDEX,
        source=["name"],
        suggest={
            "s": {
                "prefix": "signifog",
                "completion": {
                    "field": "suggest",
                    "size": 3,
                    "fuzzy": {"fuzziness": "AUTO"},
                },
            }
        },
    )
)
print("completion with fuzzy: signifog")
for o in r["suggest"]["s"][0]["options"]:
    print("   ", o["_source"]["name"][:70])

run(
    "fuzzy: Bonnie Fontana", fuzzy("Bonnie Fontana"), fields=("name", "sku", "doc_type")
)
run(
    "phonetic: Bonnie Fontana",
    phonetic("Bonnie Fontana"),
    fields=("name", "sku", "doc_type"),
)

In [ ]:
def took_ms(cfg_or_body, q=None):
    if q is not None:
        r, _ = search_one(cfg_or_body, q)
    else:
        r = retry(lambda: es.search(index=INDEX, size=K, source=False, **cfg_or_body))
    return r["took"]


def single_field(field, q, **opts):
    return {"query": {"match": {field: {"query": q, "fuzziness": "AUTO", **opts}}}}


print("Fuzzy settings on a single-field match, one request at a time")
print(f"{'setting':22} {'SKU catch':>10} {'drug catch':>11} {'drug p50 ms':>12}")
for label, opts in (
    ("default", {}),
    ("max_expansions: 10", {"max_expansions": 10}),
    ("prefix_length: 2", {"prefix_length": 2}),
):
    res = {}
    for set_name, field in (("sku_typo", "sku.text"), ("drug_typo", "name")):
        qs = [q for _, q in queries if _ == set_name]
        got, took = 0, []
        for q in qs:
            r = retry(
                lambda: es.search(
                    index=INDEX,
                    size=K,
                    source=False,
                    **single_field(field, q["query"], **opts),
                )
            )
            got += any(ratings[q["id"]].get(h["_id"], 0) > 0 for h in r["hits"]["hits"])
            took.append(r["took"])
        res[set_name] = (got / len(qs), statistics.median(took))
    print(
        f"{label:22} {res['sku_typo'][0]:10.1%} {res['drug_typo'][0]:11.1%} {res['drug_typo'][1]:12.0f}"
    )

## 8. Measure where the storage goes

`_disk_usage` (experimental) after a force merge to one segment. It's slow on a large index.

In [ ]:
es.options(request_timeout=7200).indices.forcemerge(
    index=INDEX, max_num_segments=1, wait_for_completion=True
)
du = es.options(request_timeout=7200).indices.disk_usage(
    index=INDEX, run_expensive_tasks=True
)[INDEX]
fields = du["fields"]
mb = lambda f: fields.get(f, {}).get("total_in_bytes", 0) / 2**20
jina = (
    sum(
        v["total_in_bytes"] for k, v in fields.items() if k.startswith("semantic_dense")
    )
    / 2**20
)
sizes = (
    pd.Series(
        {
            "name (reference)": mb("name"),
            "name.phonetic": mb("name.phonetic"),
            "suggest (completion)": mb("suggest"),
            "name.edge + sku.edge": mb("name.edge") + mb("sku.edge"),
            "semantic_dense (all Jina fields)": jina,
        }
    )
    .round(1)
    .rename("MiB")
)
print(sizes.to_string())
print(
    f"whole index: {du['store_size_in_bytes'] / 2**30:.2f} GiB | Jina fields: {jina / 1024:.2f} GiB"
)
print("\nLargest fields:")
for k, v in sorted(fields.items(), key=lambda kv: -kv[1]["total_in_bytes"])[:8]:
    print(f"  {k:55} {v['total_in_bytes'] / 2**20:10,.1f} MiB")

## 9. Run semantic search on misspelled queries

![Approximate kNN only scores the documents it visits (illustration; the scores are real)](images/hnsw-candidate-pool.jpg)

In [ ]:
def expected_scores(q, ids):
    body = {
        "query": {
            "knn": {
                "field": "semantic_dense",
                "k": len(ids),
                "num_candidates": max(len(ids), 10),
                "query_vector_builder": {"text_embedding": {"model_text": q}},
                "filter": {"ids": {"values": list(ids)}},
            }
        }
    }
    r = retry(lambda: es.search(index=INDEX, size=len(ids), source=["name"], **body))
    return [
        (round(h["_score"], 4), h["_source"]["name"][:60]) for h in r["hits"]["hits"]
    ]


run("knn: wireles headphnes", semantic_knn("wireles headphnes"))
run("knn: signifog", semantic_knn("signifog"))
run("knn: lanoxjn", semantic_knn("lanoxjn"))

run("default match on semantic_dense: macrobid", semantic_match("macrobid"), n=2)
run("knn, num_candidates 10000: macrobid", semantic_knn("macrobid"), n=2)
macrobid = [d["_id"] for d in medications if "macrobid" in d["name"].lower()]
print(
    "Macrobid documents scored directly:",
    expected_scores("macrobid", macrobid) if macrobid else "not in this catalog",
)

In [ ]:
# Expected document vs the closest other document, on a sample of typo queries per set.
sample_n = 60 if SCALE == "article" else 30
print(
    f"Share of typo queries where an expected document outscores every other document ({sample_n} per set)"
)
for s in ("sku_typo", "drug_typo", "surname_typo", "surname_soundalike"):
    qs = [q for _, q in queries if _ == s][:sample_n]
    wins = 0
    for q in qs:
        best_expected = max(
            (sc for sc, _ in expected_scores(q["query"], q["ratings"])), default=0
        )
        r = retry(
            lambda: es.search(
                index=INDEX, size=K, source=False, **semantic_knn(q["query"])
            )
        )
        others = [
            h["_score"] for h in r["hits"]["hits"] if h["_id"] not in q["ratings"]
        ]
        wins += best_expected > (max(others) if others else 0)
    print(f"  {s:20} {wins / len(qs):6.1%}")

## 10. Compare synonyms on listed and unlisted misspellings

In [ ]:
phr = [q for q in ratings if set_of[q] == "word_typo"]
print(
    "Misspelled title phrases: catch rate by whether the misspelling is in the synonyms set"
)
for cfg in ("synonyms", "fuzzy", "semantic_knn"):
    listed = statistics.mean(caught(cfg, q) for q in phr if meta[q]["split"] == "train")
    unlisted = statistics.mean(
        caught(cfg, q) for q in phr if meta[q]["split"] == "test"
    )
    print(f"  {cfg:14} listed {listed:6.1%} | unlisted {unlisted:6.1%}")

run(
    "synonyms analyzer: bose wireless reciever",
    {
        "query": {
            "match": {
                "name": {
                    "query": "bose wireless reciever",
                    "operator": "and",
                    "analyzer": "with_misspellings",
                }
            }
        }
    },
)

## 11. Combine fuzzy and semantic search

![Four ways to combine exact matching, fuzzy and semantic search](images/combinations.jpg)

In [ ]:
exact_q = [
    q
    for q in ratings
    if set_of[q] in ("word_exact", "drug_exact", "surname_exact", "sku_exact")
]
fell_back = sum(not hits["strict"][q] for q in exact_q)
print(
    f"correctly spelled twins: {len(exact_q)} | the fallback ran fuzzy for {fell_back} of them"
)
real = [q for q in ratings if set_of[q] == "word_typo_real"]
print(
    f"real shopper typo queries where strict already finds something: {statistics.mean(bool(hits['strict'][q]) for q in real):.1%}"
)

for cfg in ("rrf", "rrf_rerank_name", "rrf_rerank"):
    sku_exact = [q for q in exact_q if set_of[q] == "sku_exact"]
    print(
        f"{cfg:16} wrong first on correctly spelled SKUs: {statistics.mean(wrong_first(cfg, q) for q in sku_exact):6.1%}"
    )

run(
    "RRF + reranker reading name: celexa",
    reranked("name")("celexa"),
    fields=("name", "sku", "doc_type"),
)
run(
    "RRF + reranker reading name and SKU: celexa",
    reranked("label")("celexa"),
    fields=("name", "sku", "doc_type"),
)

The same hybrid in ES|QL (`FUSE` needs 9.5):

In [ ]:
ESQL = """
FROM typo-catalog METADATA _id, _index, _score
| FORK
    ( WHERE MATCH(name, "buisness card laminating sheets", {"fuzziness": "AUTO", "operator": "AND"})
      | SORT _score DESC | LIMIT 50 )
    ( WHERE KNN(semantic_dense, TEXT_EMBEDDING("buisness card laminating sheets", ".jina-embeddings-v5-text-small"), {"min_candidates": 10000})
      | SORT _score DESC | LIMIT 50 )
| FUSE
| SORT _score DESC
| LIMIT 50
| RERANK "buisness card laminating sheets" ON name, sku WITH {"inference_id": ".jina-reranker-v3.5"}
| SORT _score DESC
| EVAL name = LEFT(name, 48)
| KEEP _fork, name
| LIMIT 5
"""
print(retry(lambda: es.esql.query(query=ESQL, format="txt")).body)

## 12. Measure what each combination costs

Latency is the `took` value, one request at a time.

In [ ]:
exact_sets = ["word_exact", "drug_exact", "surname_exact", "sku_exact"]
n_exact = {s: int(results[(results.set == s)].n.iloc[0]) for s in exact_sets}
wrong = results[results.set.isin(exact_sets)].assign(
    w=lambda d: d.wrong_first * d.set.map(n_exact)
)
costs = pd.DataFrame(
    {
        "wrong_first_on_exact_%": wrong.groupby("config").w.sum()
        / sum(n_exact.values())
        * 100,
        "false_positives_%": results[results.set == "negative"]
        .set_index("config")
        .false_positives
        * 100,
        "exploratory_ndcg@10": results[results.set == "exploratory"]
        .set_index("config")
        .ndcg,
    }
).loc[ORDER]
print(f"correctly spelled queries: {sum(n_exact.values())} | negatives: {len(neg)}")
print(
    costs.round(
        {"wrong_first_on_exact_%": 1, "false_positives_%": 1, "exploratory_ndcg@10": 2}
    ).to_string()
)

In [ ]:
lat_queries = [q for s in TYPO_SETS for q in [x for _, x in queries if _ == s][:40]]
lat = {}
for cfg in [
    "strict",
    "fuzzy",
    "edge_ngram",
    "phonetic",
    "completion",
    "synonyms",
    "semantic_match",
    "semantic_knn",
    "blend",
    "rrf",
    "rrf_rerank_name",
    "rrf_rerank",
]:
    took = sorted(search_one(cfg, q["query"])[0]["took"] for q in lat_queries)
    lat[cfg] = {
        "p50_ms": statistics.median(took),
        "p95_ms": took[int(0.95 * (len(took) - 1))],
    }
print(f"latency over {len(lat_queries)} typo queries, one request at a time")
print(pd.DataFrame(lat).T.to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
show_cfg = ["fuzzy", "semantic_knn", "rrf", "rrf_rerank"]
catch.loc[show_cfg].T.plot.bar(
    ax=axes[0], color=["#0B64DD", "#48EFCF", "#FF957D", "#F04E98"], width=0.8
)
axes[0].set_title("Catch rate (relevant result in the top 10)")
axes[0].set_ylim(0, 1.05)
axes[0].tick_params(axis="x", rotation=30)
costs.loc[show_cfg, ["wrong_first_on_exact_%", "false_positives_%"]].plot.barh(
    ax=axes[1], color=["#0B64DD", "#FF957D"]
)
axes[1].set_title("Costs, %")
axes[1].invert_yaxis()
plt.tight_layout()
plt.show()

## 13. Clean up

Deletes the index (only if this notebook created it) and the synonyms set. Uncomment the last line to also delete `nb_data/`.

In [ ]:
if index_is_ours():
    es.indices.delete(index=INDEX)
es.options(ignore_status=404).synonyms.delete_synonym(id=SYNONYMS_SET)
print("index exists:", bool(es.indices.exists(index=INDEX)))
# import shutil; shutil.rmtree("nb_data")